# PCA vs DAS Rotation Quality (Single Variable)

This notebook compares two intervention parameterizations on the same base-10 addition task and the same pretrained MLP:

- **PCA intervention**: fixed rotated basis, swap in the first `|s|` PCA coordinates.
- **DAS intervention**: learned rotation, swap in a fixed subspace length `|s|` (location is implicit under learned rotation).

Evaluation metric: held-out counterfactual accuracy for one selected abstract variable.


In [ ]:
import os
import random
import itertools

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from pyvene import CausalModel
from pyvene import IntervenableModel
from pyvene import RepresentationConfig, IntervenableConfig
from pyvene import PCARotatedSpaceIntervention, RotatedSpaceIntervention, VanillaIntervention

from variable_width_mlp import load_variable_width_mlp_checkpoint, logits_from_output

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = 'cpu'
print("Using device:", device)


## Hyperparameters


In [ ]:
# Model checkpoint candidates (first existing path is used).
CHECKPOINT_CANDIDATES = [
    "artifacts/addition_mlp.pt",
]

# Abstract variable to evaluate.
TARGET_VAR = "C1"  # one of: "S1", "C1", "S2", "C2"

# Intervention location / subspace size.
ALIGN_LAYER = 0 # 0-indexed
SUBSPACE_DIM = 8
COMPONENT = None   # if None -> use explicit path f"h[{ALIGN_LAYER}].output"
UNIT = "pos"
MAX_UNITS = 1
INTERVENTION_POS = 0

# Dataset sizes.
CF_TRAIN_SIZE = 12000
CF_TEST_SIZE = 3000
CF_BATCH_SIZE = 128
PCA_FIT_FACTUAL_SIZE = 15000

# DAS optimization.
DAS_EPOCHS = 8
DAS_LR = 1e-3

print("TARGET_VAR:", TARGET_VAR)
print("ALIGN_LAYER:", ALIGN_LAYER)
print("SUBSPACE_DIM:", SUBSPACE_DIM)


## SCM Definition (Base-10 Two-Digit Addition)


In [ ]:
one_hot_digits = [np.eye(10, dtype=np.float32)[d] for d in range(10)]


def as_digit(x):
    arr = np.array(x).reshape(-1)
    if arr.size == 1:
        return int(arr[0])
    return int(arr.argmax())


variables = ["A1", "B1", "A2", "B2", "S1", "C1", "T2", "S2", "C2", "O"]
values = {
    "A1": one_hot_digits,
    "B1": one_hot_digits,
    "A2": one_hot_digits,
    "B2": one_hot_digits,
    "S1": list(range(10)),
    "C1": [0, 1],
    "T2": list(range(19)),
    "S2": list(range(10)),
    "C2": [0, 1],
    "O": list(range(200)),  # includes 199 for intervention-consistent states
}

parents = {
    "A1": [],
    "B1": [],
    "A2": [],
    "B2": [],
    "S1": ["A1", "B1"],
    "C1": ["A1", "B1"],
    "T2": ["A2", "B2"],
    "S2": ["T2", "C1"],
    "C2": ["T2", "C1"],
    "O": ["C2", "S2", "S1"],
}


def FILLER():
    return one_hot_digits[0]


functions = {
    "A1": FILLER,
    "B1": FILLER,
    "A2": FILLER,
    "B2": FILLER,
    "S1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) % 10,
    "C1": lambda a1, b1: (as_digit(a1) + as_digit(b1)) // 10,
    "T2": lambda a2, b2: as_digit(a2) + as_digit(b2),
    "S2": lambda t2, c1: (int(t2) + int(c1)) % 10,
    "C2": lambda t2, c1: (int(t2) + int(c1)) // 10,
    "O": lambda c2, s2, s1: int(100 * int(c2) + 10 * int(s2) + int(s1)),
}

addition_model = CausalModel(variables, values, parents, functions)
print("SCM ready.")


## Load Pretrained MLP


In [ ]:
MODEL_CHECKPOINT_PATH = None
for p in CHECKPOINT_CANDIDATES:
    if os.path.exists(p):
        MODEL_CHECKPOINT_PATH = p
        break
if MODEL_CHECKPOINT_PATH is None:
    raise FileNotFoundError(f"No checkpoint found in {CHECKPOINT_CANDIDATES}")

trained, loaded_cfg, checkpoint = load_variable_width_mlp_checkpoint(MODEL_CHECKPOINT_PATH, device)
trained.eval()

if ALIGN_LAYER < 0 or ALIGN_LAYER >= loaded_cfg.n_layer:
    raise ValueError(f"ALIGN_LAYER={ALIGN_LAYER} out of range for n_layer={loaded_cfg.n_layer}")

layer_hidden_dim = int(loaded_cfg.hidden_dims[ALIGN_LAYER])
if SUBSPACE_DIM <= 0 or SUBSPACE_DIM > layer_hidden_dim:
    raise ValueError(f"SUBSPACE_DIM must be in [1,{layer_hidden_dim}], got {SUBSPACE_DIM}")

component_path = COMPONENT if COMPONENT is not None else f"h[{ALIGN_LAYER}].output"

print("Loaded checkpoint:", MODEL_CHECKPOINT_PATH)
print("Hidden dims:", loaded_cfg.hidden_dims)
print("Using component:", component_path)


## Counterfactual Data for One Variable


In [ ]:
VAR_VALUE_SPACE = {
    "S1": list(range(10)),
    "C1": [0, 1],
    "S2": list(range(10)),
    "C2": [0, 1],
}
if TARGET_VAR not in VAR_VALUE_SPACE:
    raise ValueError(f"Unsupported TARGET_VAR {TARGET_VAR}")


def digit_assignment(a1, b1, a2, b2):
    return {
        "A1": one_hot_digits[int(a1)],
        "B1": one_hot_digits[int(b1)],
        "A2": one_hot_digits[int(a2)],
        "B2": one_hot_digits[int(b2)],
    }


all_assignments = [
    digit_assignment(a1, b1, a2, b2)
    for a1, b1, a2, b2 in itertools.product(range(10), repeat=4)
]

bucket_by_target = {val: [] for val in VAR_VALUE_SPACE[TARGET_VAR]}
for assignment in all_assignments:
    setting = addition_model.run_forward(assignment)
    bucket_by_target[int(setting[TARGET_VAR])].append(assignment)


def cf_intervention_id(intervention):
    # Single target variable for this notebook.
    return 0


def cf_input_sampler(*args, **kwargs):
    output_var = kwargs.get("output_var", None)
    output_var_value = kwargs.get("output_var_value", None)

    if output_var is None:
        return random.choice(all_assignments)

    if output_var != TARGET_VAR:
        return random.choice(all_assignments)

    return random.choice(bucket_by_target[int(output_var_value)])


def cf_intervention_sampler(*args, **kwargs):
    return {TARGET_VAR: random.choice(VAR_VALUE_SPACE[TARGET_VAR])}


train_cf_dataset = addition_model.generate_counterfactual_dataset(
    CF_TRAIN_SIZE,
    cf_intervention_id,
    CF_BATCH_SIZE,
    device=device,
    sampler=cf_input_sampler,
    intervention_sampler=cf_intervention_sampler,
)

test_cf_dataset = addition_model.generate_counterfactual_dataset(
    CF_TEST_SIZE,
    cf_intervention_id,
    CF_BATCH_SIZE,
    device=device,
    sampler=cf_input_sampler,
    intervention_sampler=cf_intervention_sampler,
)

# Safety guard: if the checkpoint has fewer output classes than the SCM labels,
# filter out out-of-range CF examples so loss/one-hot computations do not crash.
model_num_classes = int(getattr(trained.score, "out_features", trained.config.num_classes))


def filter_cf_dataset_by_label_range(dataset, num_classes):
    valid_indices = []
    dropped = 0
    for idx in range(len(dataset)):
        label = int(dataset[idx]["labels"].item())
        if 0 <= label < num_classes:
            valid_indices.append(idx)
        else:
            dropped += 1
    subset = torch.utils.data.Subset(dataset, valid_indices) if dropped > 0 else dataset
    return subset, dropped


train_cf_dataset, dropped_train = filter_cf_dataset_by_label_range(train_cf_dataset, model_num_classes)
test_cf_dataset, dropped_test = filter_cf_dataset_by_label_range(test_cf_dataset, model_num_classes)

if dropped_train > 0 or dropped_test > 0:
    print(
        f"Filtered out-of-range labels for num_classes={model_num_classes}: "
        f"train_dropped={dropped_train}, test_dropped={dropped_test}"
    )

print("Train CF size:", len(train_cf_dataset))
print("Test CF size:", len(test_cf_dataset))
print("Example keys:", train_cf_dataset[0].keys())
print("Example target var:", TARGET_VAR)


## Shared Intervention Helpers


In [ ]:
import pyvene.models.modeling_utils as mu
from pyvene.models.mlp.modelings_mlp import MLPForClassification as PV_MLP


def register_custom_model_with_pyvene(custom_cls, base_cls=PV_MLP):
    """Map custom model class to pyvene's built-in MLP metadata registries."""
    touched = []
    for name, obj in vars(mu).items():
        if name.startswith("type_to_") and isinstance(obj, dict) and base_cls in obj:
            if custom_cls not in obj:
                obj[custom_cls] = obj[base_cls]
            touched.append(name)
    return touched


def collect_layer_activations(model, inputs_flat, layer_idx, batch_size=256):
    acts = []
    n = inputs_flat.shape[0]
    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            hidden = inputs_flat[start:end].to(device).unsqueeze(1)
            for i, block in enumerate(model.h):
                hidden = block(hidden)
                if i == layer_idx:
                    acts.append(hidden[:, 0, :].detach().cpu())
                    break
    return torch.cat(acts, dim=0).numpy()


def layer_hidden_dim(model, layer_idx):
    if hasattr(model.config, "hidden_dims"):
        return int(model.config.hidden_dims[layer_idx])
    return int(loaded_cfg.hidden_dims[layer_idx])


def make_single_rep_intervenable(model, layer_idx, component, intervention, use_fast=False):
    # Ensure pyvene can resolve dimension/hooks for the custom model class.
    touched = register_custom_model_with_pyvene(type(model))

    # pyvene MLP mappings sometimes read config.h_dim.
    hidden_dim = layer_hidden_dim(model, layer_idx)
    model.config.h_dim = hidden_dim

    cfg_int = IntervenableConfig(
        model_type=type(model),
        representations=[
            RepresentationConfig(
                layer=layer_idx,
                component=component,
                unit=UNIT,
                max_number_of_units=MAX_UNITS,
                intervention=intervention,
            )
        ],
    )
    intervenable = IntervenableModel(cfg_int, model, use_fast=use_fast)
    intervenable.set_device(device)
    intervenable.disable_model_gradients()

    if len(touched) > 0:
        print(f"Registered custom model in {len(touched)} pyvene maps for layer={layer_idx}")
    return intervenable


def make_single_rep_das_intervenable(model, layer_idx, component, use_fast=False):
    hidden_dim = layer_hidden_dim(model, layer_idx)
    das_intervention = RotatedSpaceIntervention(embed_dim=hidden_dim)
    return make_single_rep_intervenable(
        model=model,
        layer_idx=layer_idx,
        component=component,
        intervention=das_intervention,
        use_fast=use_fast,
    )


def run_cf_batch_single(intervenable, batch, subspace_dims):
    bsz = batch["input_ids"].shape[0]

    base_batch = batch["input_ids"]
    if base_batch.ndim == 2:
        base_batch = base_batch.unsqueeze(1)

    source_raw = batch["source_input_ids"]
    if source_raw.ndim == 2:
        source_batch = source_raw.unsqueeze(1)
    elif source_raw.ndim == 3:
        source_batch = source_raw[:, 0]
        if source_batch.ndim == 2:
            source_batch = source_batch.unsqueeze(1)
    else:
        raise ValueError(f"Unexpected source_input_ids shape: {tuple(source_raw.shape)}")

    pos_list = [[INTERVENTION_POS]] * bsz
    subspaces = [list(subspace_dims)] * bsz

    _, cf_outputs = intervenable(
        {"inputs_embeds": base_batch},
        [{"inputs_embeds": source_batch}],
        {"sources->base": ([pos_list], [pos_list])},
        subspaces=[subspaces],
    )
    return logits_from_output(cf_outputs)


def evaluate_intervenable(intervenable, dataset, subspace_dims, batch_size=CF_BATCH_SIZE):
    labels_all, preds_all = [], []
    loader = DataLoader(dataset, batch_size=batch_size)
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(device)

            logits = run_cf_batch_single(intervenable, batch, subspace_dims)
            preds = torch.argmax(logits, dim=1)
            gold = batch["labels"].to(torch.long).view(-1)

            labels_all.append(gold.detach().cpu())
            preds_all.append(preds.detach().cpu())

    labels_all = torch.cat(labels_all)
    preds_all = torch.cat(preds_all)
    acc = float((labels_all == preds_all).to(torch.float32).mean().item())
    return acc


def train_das_intervention(intervenable_das, dataset, subspace_dims, epochs, lr, batch_size):
    optimizer_params = []
    for intervention in intervenable_das.interventions.values():
        if hasattr(intervention, "rotate_layer"):
            optimizer_params.append({"params": intervention.rotate_layer.parameters()})

    if len(optimizer_params) == 0:
        raise RuntimeError("No rotate_layer parameters found for DAS intervention.")

    optimizer = torch.optim.Adam(optimizer_params, lr=lr)

    intervenable_das.model.train()
    for epoch in range(epochs):
        losses = []
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
        for batch in loader:
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(device)

            logits = run_cf_batch_single(intervenable_das, batch, subspace_dims)
            labels = batch["labels"].to(torch.long).view(-1)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.detach().cpu()))

        print(f"DAS epoch {epoch + 1}/{epochs} mean loss: {np.mean(losses):.6f}")

    intervenable_das.model.eval()


## PCA Intervention (Fixed Rotation)


In [ ]:
# Fit PCA on factual activations from the selected layer.

def factual_input_sampler():
    return random.choice(all_assignments)


pca_examples = addition_model.generate_factual_dataset(PCA_FIT_FACTUAL_SIZE, factual_input_sampler)
pca_inputs = torch.stack([ex["input_ids"] for ex in pca_examples]).to(torch.float32)

layer_acts = collect_layer_activations(trained, pca_inputs, ALIGN_LAYER)
pca_mean = layer_acts.mean(axis=0)
pca_std = layer_acts.std(axis=0) + 1e-6

pca = PCA(n_components=layer_hidden_dim(trained, ALIGN_LAYER), svd_solver="full")
pca.fit((layer_acts - pca_mean) / pca_std)

pca_intervention = PCARotatedSpaceIntervention(
    embed_dim=layer_hidden_dim(trained, ALIGN_LAYER),
    pca=pca,
    pca_mean=pca_mean,
    pca_std=pca_std,
)

intervenable_pca = make_single_rep_intervenable(
    model=trained,
    layer_idx=ALIGN_LAYER,
    component=component_path,
    intervention=pca_intervention,
    use_fast=False,
)
intervenable_pca.disable_intervention_gradients()

subspace_dims = list(range(SUBSPACE_DIM))
print("PCA subspace dims:", subspace_dims)


## DAS Intervention (Learned Rotation)


In [ ]:
intervenable_das = make_single_rep_das_intervenable(
    model=trained,
    layer_idx=ALIGN_LAYER,
    component=component_path,
    use_fast=False,
)

train_das_intervention(
    intervenable_das,
    train_cf_dataset,
    subspace_dims,
    epochs=DAS_EPOCHS,
    lr=DAS_LR,
    batch_size=CF_BATCH_SIZE,
)


In [ ]:
pca_acc = evaluate_intervenable(intervenable_pca, test_cf_dataset, subspace_dims, batch_size=CF_BATCH_SIZE)
das_acc = evaluate_intervenable(intervenable_das, test_cf_dataset, subspace_dims, batch_size=CF_BATCH_SIZE)

print(f"TARGET_VAR = {TARGET_VAR}")
print(f"Layer = {ALIGN_LAYER}, |s| = {SUBSPACE_DIM}")
print(f"PCA CF accuracy: {pca_acc:.4f}")
print(f"DAS CF accuracy: {das_acc:.4f}")
print(f"DAS - PCA: {das_acc - pca_acc:+.4f}")

In [ ]:
labels = ["PCA", "DAS"]
vals = [pca_acc, das_acc]

plt.figure(figsize=(4.5, 3.5))
plt.bar(labels, vals)
plt.ylim(0.0, 1.0)
plt.ylabel("Counterfactual accuracy")
plt.title(f"{TARGET_VAR} at L{ALIGN_LAYER}, |s|={SUBSPACE_DIM}")
for i, v in enumerate(vals):
    plt.text(i, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()


## Notes

- This notebook isolates one abstract variable at a time (`TARGET_VAR`) and compares intervention quality under fixed `ALIGN_LAYER` and `SUBSPACE_DIM`.
- To sweep hyperparameters, edit the hyperparameter cell and rerun from there.
- If you want a multi-seed comparison, wrap the dataset generation + DAS training + evaluation blocks in a seed loop and report mean/std.


# Sweep Over (Layer, Subspace Size)

This section compares PCA vs DAS across multiple `(layer, |s|)` pairs for the same `TARGET_VAR`.


In [ ]:
# Sweep hyperparameters.
SWEEP_LAYERS = list(range(loaded_cfg.n_layer))   # e.g. [0,1,2]
SWEEP_SUBSPACE_SIZES = [1, 16, 32, 48, 64, 80, 96, 112, 128]

SWEEP_DAS_EPOCHS = 20
SWEEP_DAS_LR = DAS_LR

# Optional speed limits for the sweep.
SWEEP_MAX_TRAIN_EXAMPLES = 4000   # None for full train_cf_dataset
SWEEP_MAX_TEST_EXAMPLES = 2000    # None for full test_cf_dataset
SWEEP_PCA_FIT_FACTUAL_SIZE = 12000

print("SWEEP_LAYERS:", SWEEP_LAYERS)
print("SWEEP_SUBSPACE_SIZES:", SWEEP_SUBSPACE_SIZES)


In [ ]:
def maybe_subset_dataset(dataset, max_examples, seed):
    if max_examples is None or len(dataset) <= max_examples:
        return dataset
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(dataset), size=max_examples, replace=False).tolist()
    return torch.utils.data.Subset(dataset, idx)


In [ ]:
# Prepare shared sweep datasets.
sweep_train_dataset = maybe_subset_dataset(train_cf_dataset, SWEEP_MAX_TRAIN_EXAMPLES, SEED + 101)
sweep_test_dataset = maybe_subset_dataset(test_cf_dataset, SWEEP_MAX_TEST_EXAMPLES, SEED + 202)

print("Sweep train size:", len(sweep_train_dataset))
print("Sweep test size:", len(sweep_test_dataset))

# Shared factual activations for PCA fitting.
def factual_input_sampler():
    return random.choice(all_assignments)


sweep_pca_examples = addition_model.generate_factual_dataset(SWEEP_PCA_FIT_FACTUAL_SIZE, factual_input_sampler)
sweep_pca_inputs = torch.stack([ex["input_ids"] for ex in sweep_pca_examples]).to(torch.float32)

# Fit one PCA basis per layer once.
pca_by_layer = {}
for layer_idx in SWEEP_LAYERS:
    hidden_dim = int(loaded_cfg.hidden_dims[layer_idx])
    acts = collect_layer_activations(trained, sweep_pca_inputs, layer_idx)
    mean = acts.mean(axis=0)
    std = acts.std(axis=0) + 1e-6
    pca = PCA(n_components=hidden_dim, svd_solver="full")
    pca.fit((acts - mean) / std)
    pca_by_layer[layer_idx] = {
        "hidden_dim": hidden_dim,
        "pca": pca,
        "mean": mean,
        "std": std,
    }
    print(f"Fitted PCA for layer {layer_idx} (hidden_dim={hidden_dim})")

results = []
vanilla_results = []
for layer_idx in SWEEP_LAYERS:
    hidden_dim = pca_by_layer[layer_idx]["hidden_dim"]
    component = f"h[{layer_idx}].output"

    for sdim in SWEEP_SUBSPACE_SIZES:
        if sdim > hidden_dim:
            print(f"Skip layer={layer_idx}, |s|={sdim} (hidden_dim={hidden_dim})")
            continue

        print(f"\n=== layer={layer_idx}, |s|={sdim} ===")
        subspace_dims = list(range(sdim))

        # 1) PCA intervention at this layer.
        pca_int = PCARotatedSpaceIntervention(
            embed_dim=hidden_dim,
            pca=pca_by_layer[layer_idx]["pca"],
            pca_mean=pca_by_layer[layer_idx]["mean"],
            pca_std=pca_by_layer[layer_idx]["std"],
        )
        int_pca = make_single_rep_intervenable(
            model=trained,
            layer_idx=layer_idx,
            component=component,
            intervention=pca_int,
            use_fast=False,
        )
        int_pca.disable_intervention_gradients()
        pca_acc = evaluate_intervenable(int_pca, sweep_test_dataset, subspace_dims, batch_size=CF_BATCH_SIZE)
        print(f"PCA acc: {pca_acc:.4f}")

        # 2) Learned DAS rotation at this layer.
        int_das = make_single_rep_das_intervenable(
            model=trained,
            layer_idx=layer_idx,
            component=component,
            use_fast=False,
        )
        train_das_intervention(
            int_das,
            sweep_train_dataset,
            subspace_dims,
            epochs=SWEEP_DAS_EPOCHS,
            lr=SWEEP_DAS_LR,
            batch_size=CF_BATCH_SIZE,
        )
        das_acc = evaluate_intervenable(int_das, sweep_test_dataset, subspace_dims, batch_size=CF_BATCH_SIZE)
        print(f"DAS acc: {das_acc:.4f}")

        # 3) Direct swap in the original hidden basis, with no rotation at all.
        int_vanilla = make_single_rep_intervenable(
            model=trained,
            layer_idx=layer_idx,
            component=component,
            intervention=VanillaIntervention(),
            use_fast=False,
        )
        int_vanilla.disable_intervention_gradients()
        vanilla_acc = evaluate_intervenable(
            int_vanilla,
            sweep_test_dataset,
            subspace_dims,
            batch_size=CF_BATCH_SIZE,
        )
        print(f"Direct-swap acc: {vanilla_acc:.4f}")

        vanilla_results.append(
            {
                "layer": int(layer_idx),
                "subspace_dim": int(sdim),
                "vanilla_acc": float(vanilla_acc),
            }
        )

        results.append(
            {
                "layer": int(layer_idx),
                "subspace_dim": int(sdim),
                "pca_acc": float(pca_acc),
                "das_acc": float(das_acc),
                "vanilla_acc": float(vanilla_acc),
                "delta": float(das_acc - pca_acc),
            }
        )

print("\nSweep finished. Num evaluated pairs:", len(results))


In [ ]:
results

In [ ]:
# Plot sweep results with numeric annotations in each cell.
if len(results) == 0:
    raise RuntimeError("No valid (layer, subspace) pairs were evaluated.")

plot_rows = [r for r in results if "vanilla_acc" in r]
if len(plot_rows) == 0:
    raise RuntimeError("Rerun the sweep cell so direct-swap results are available.")

layers_sorted = sorted({r["layer"] for r in plot_rows})
subs_sorted = sorted({r["subspace_dim"] for r in plot_rows})

layer_to_i = {l: i for i, l in enumerate(layers_sorted)}
sub_to_j = {s: j for j, s in enumerate(subs_sorted)}

pca_mat = np.full((len(layers_sorted), len(subs_sorted)), np.nan, dtype=np.float32)
das_mat = np.full_like(pca_mat, np.nan)
vanilla_mat = np.full_like(pca_mat, np.nan)

for r in plot_rows:
    i = layer_to_i[r["layer"]]
    j = sub_to_j[r["subspace_dim"]]
    pca_mat[i, j] = r["pca_acc"]
    das_mat[i, j] = r["das_acc"]
    vanilla_mat[i, j] = r["vanilla_acc"]


def annotate_heatmap(ax, mat, fmt="{:.3f}"):
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat[i, j]
            if np.isnan(val):
                continue
            text_color = "white" if val < 0.45 else "black"
            ax.text(
                j,
                i,
                fmt.format(val),
                ha="center",
                va="center",
                color=text_color,
                fontsize=9,
            )


fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

im0 = axes[0].imshow(vanilla_mat, aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
axes[0].set_title("Direct Swap CF Accuracy")
axes[0].set_xlabel("|s|")
axes[0].set_ylabel("Layer")
axes[0].set_xticks(range(len(subs_sorted)), subs_sorted)
axes[0].set_yticks(range(len(layers_sorted)), layers_sorted)
annotate_heatmap(axes[0], vanilla_mat)
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(pca_mat, aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
axes[1].set_title("PCA CF Accuracy")
axes[1].set_xlabel("|s|")
axes[1].set_ylabel("Layer")
axes[1].set_xticks(range(len(subs_sorted)), subs_sorted)
axes[1].set_yticks(range(len(layers_sorted)), layers_sorted)
annotate_heatmap(axes[1], pca_mat)
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

im2 = axes[2].imshow(das_mat, aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
axes[2].set_title("DAS CF Accuracy")
axes[2].set_xlabel("|s|")
axes[2].set_ylabel("Layer")
axes[2].set_xticks(range(len(subs_sorted)), subs_sorted)
axes[2].set_yticks(range(len(layers_sorted)), layers_sorted)
annotate_heatmap(axes[2], das_mat)
fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

plt.suptitle(f"TARGET_VAR={TARGET_VAR}: PCA vs DAS vs Direct Swap", y=1.03)
plt.show()

print("Best PCA setting:")
print(max(plot_rows, key=lambda x: x["pca_acc"]))

print("\nBest DAS setting:")
print(max(plot_rows, key=lambda x: x["das_acc"]))

print("\nBest direct-swap setting:")
print(max(plot_rows, key=lambda x: x["vanilla_acc"]))


In [ ]:
# Line plots for sweep results: x-axis is |s|, one line per layer.
if len(results) == 0:
    raise RuntimeError("No valid (layer, subspace) pairs were evaluated.")

plot_rows = [r for r in results if "vanilla_acc" in r]
if len(plot_rows) == 0:
    raise RuntimeError("Rerun the sweep cell so direct-swap results are available.")

layers_sorted = sorted({r["layer"] for r in plot_rows})
subs_sorted = sorted({r["subspace_dim"] for r in plot_rows})

methods = [
    ("vanilla_acc", "Direct Swap CF Accuracy"),
    ("pca_acc", "PCA CF Accuracy"),
    ("das_acc", "DAS CF Accuracy"),
]

# Build lookup: scores[(method, layer, subspace_dim)] = accuracy
scores = {}
for r in plot_rows:
    for method_key, _ in methods:
        scores[(method_key, r["layer"], r["subspace_dim"])] = r[method_key]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True, constrained_layout=True)
colors = plt.cm.tab10(np.linspace(0, 1, len(layers_sorted)))

for ax, (method_key, title) in zip(axes, methods):
    for color, layer in zip(colors, layers_sorted):
        y = [scores.get((method_key, layer, s), np.nan) for s in subs_sorted]
        ax.plot(subs_sorted, y, marker="o", linewidth=2, color=color, label=f"Layer {layer}")

        # Annotate each point.
        for x_val, y_val in zip(subs_sorted, y):
            if not np.isnan(y_val):
                ax.text(x_val, y_val + 0.015, f"{y_val:.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_title(title)
    ax.set_xlabel("|s|")
    ax.set_xticks(subs_sorted)
    ax.set_ylim(0.0, 1.0)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Counterfactual accuracy")
axes[0].legend(title="Layer", loc="best")

plt.suptitle(f"TARGET_VAR={TARGET_VAR}: PCA vs DAS vs Direct Swap", y=1.03)
plt.show()

print("Best PCA setting:")
print(max(plot_rows, key=lambda x: x["pca_acc"]))

print("\nBest DAS setting:")
print(max(plot_rows, key=lambda x: x["das_acc"]))

print("\nBest direct-swap setting:")
print(max(plot_rows, key=lambda x: x["vanilla_acc"]))
